# Common Fund Multiple Dataset Crossing
This notebook contains the most up-to-date script to compute the crossings for the Harmonizome Cross feature. Gene set libraries for each dataset are crossed to find significant crossings by computing the Fisher's exact test p-value and Jaccard index for each pair of gene sets. Gene sets with a p<0.01 are kept, and up to 5,000 of the top-ranked crossings are displayed for each dataset pair.

In [25]:
import os
import json
from itertools import combinations
import pandas as pd
import numpy as np
from tqdm import tqdm
import pyenrichr as pye
from scipy.stats import fisher_exact
import heapq
import warnings
from numpy.random import default_rng
from scipy.stats import hypergeom
from scipy.special import gammaln
import superexacttestpy as superexact

In [26]:
import scipy.special

In [27]:
def load_gmt(file, dir=''):
    gmt = {}
    with open(file, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            if len(line.strip().split('\t'))==1: continue
            term, _, *geneset = line.strip().split('\t')
            if geneset:
                gmt[f"{term}{dir}"] = set(geneset)
    return gmt

In [28]:
_PVAL_CACHE = {}

def logcomb(n: int, k: int):
    return gammaln(n + 1) - (gammaln(k + 1) + gammaln(n - k + 1))

def get_superexact_log10p(k:int,N:int, *set_sizes: int):
    sorted_sizes = tuple(sorted(set_sizes))
    key = (k, *sorted_sizes, N)
    
    if key in _PVAL_CACHE:
        return _PVAL_CACHE[key]

    max_k = min(sorted_sizes)
    if k > max_k: return 0.0
    if k <= 0: return 0.0
    
    # Vector of intersection sizes from k to theoretical max
    i = np.arange(k, max_k + 1)
    
    # Correct log-space calculation
    log_num = logcomb(N, i)
    log_den = 0.0

    for n_m in sorted_sizes:
        log_num += logcomb(N - i, n_m - i)
        log_den += logcomb(N, n_m)
    
    ln_probs = log_num - log_den
    
    # Log-Sum-Exp trick
    max_ln = np.max(ln_probs)
    ln_pval = max_ln + np.log(np.sum(np.exp(ln_probs - max_ln)))
    
    # Convert to log10 and cache
    log10_p = min(0.0, ln_pval / np.log(10))
    _PVAL_CACHE[key] = log10_p
    return log10_p

In [29]:
def perm_pval(
    size_a, size_b, size_c,
    k_obs,
    N,
    n_perm=10000,
    seed=None
):
    rng = default_rng(seed)
    hits = 0

    for _ in range(n_perm):
        ab = hypergeom.rvs(
            M=N,
            n=size_a,
            N=size_b,
            random_state=rng
        )

        abc = hypergeom.rvs(
            M=N,
            n=ab,
            N=size_c,
            random_state=rng
        )

        if abc >= k_obs:
            hits += 1

    return (hits + 1) / (n_perm + 1)

def cross_3_gmts(
    gmt1, gmt2, gmt3,
    top_k=5000,
    background_size=22000
):
    gmt1_items = sorted(gmt1.items(), key=lambda x: len(x[1]))
    gmt2_items = sorted(gmt2.items(), key=lambda x: len(x[1]))
    gmt3_items = sorted(gmt3.items(), key=lambda x: len(x[1]))

    heap = []

    for term1, gs1 in tqdm(gmt1_items):
        for term2, gs2 in gmt2_items:

            k12 = gs1 & gs2
            if not k12:
                continue

            for term3, gs3 in gmt3_items:
                k123 = k12 & gs3
                k = len(k123)

                if not k:
                    continue

                k13 = gs1 & gs3
                k23 = gs2 & gs3

                p1 = hypergeom.logsf(k - 1, background_size, len(gs3), len(k12))
                p2 = hypergeom.logsf(k - 1, background_size, len(gs2), len(k13))
                p3 = hypergeom.logsf(k - 1, background_size, len(gs1), len(k23))
                pval = max(p1, p2, p3)

                jaccard = k / len(gs1 | gs2 | gs3)

                record = (
                    -pval,
                    jaccard,
                    k,
                    term1,
                    len(gs1),
                    term2,
                    len(gs2),
                    term3,
                    len(gs3),
                    k123,
                )

                if len(heap) < top_k:
                    heapq.heappush(heap, record)
                elif record > heap[0]:
                    heapq.heapreplace(heap, record)

    # final sorting: p-value ↑, jaccard ↓
    results = sorted(
        heap,
        key=lambda x: (-x[0], -x[1], -x[2])
    )

    return [
        #"\t".join(
            [
                term1,
                len1,
                term2,
                len2,
                term3,
                len3,
                f"{np.exp(-pval):.6e}",
                jaccard,
                str(k),
                " ".join(sorted(genes)),
            ]
        #)
        for pval, jaccard, k, term1, len1, term2, len2, term3, len3, genes in results
    ]

In [30]:
pval_cache = {}

def cross_4_gmts(
    gmt1, gmt2, gmt3, gmt4,
    top_k=5000,
    background_size=22000
):
    gmt1_items = sorted(gmt1.items(), key=lambda x: len(x[1]))
    gmt2_items = sorted(gmt2.items(), key=lambda x: len(x[1]))
    gmt3_items = sorted(gmt3.items(), key=lambda x: len(x[1]))
    gmt4_items = sorted(gmt4.items(), key=lambda x: len(x[1]))

    heap = []

    for term1, gs1 in tqdm(gmt1_items):
        for term2, gs2 in gmt2_items:

            k12 = gs1 & gs2
            if not k12:
                continue

            for term3, gs3 in gmt3_items:
                k123 = k12 & gs3
                if not k123:
                    continue

                k13 = gs1 & gs3
                k23 = gs2 & gs3

                for term4, gs4 in gmt4_items:
                    k1234 = k123 & gs4
                    k = len(k1234)

                    if not k:
                        continue

                    k124 = gs4 & k12
                    k234 = gs4 & k23
                    k134 = gs4 & k13

                    key = (k - 1, background_size, len(gs4), len(k123))
                    p1 = hypergeom.logsf(*key) if key not in pval_cache else pval_cache[key]
                    key = (k - 1, background_size, len(gs3), len(k124))
                    p2 = hypergeom.logsf(*key) if key not in pval_cache else pval_cache[key]
                    key = (k - 1, background_size, len(gs2), len(k134))
                    p3 = hypergeom.logsf(*key) if key not in pval_cache else pval_cache[key]
                    key = (k - 1, background_size, len(gs1), len(k234))
                    p4 = hypergeom.logsf(*key) if key not in pval_cache else pval_cache[key]
                    
                    pval = max(p1, p2, p3, p4)

                    jaccard = k / len(gs1 | gs2 | gs3 | gs4)

                    record = (
                        -pval,
                        jaccard,
                        k,
                        term1,
                        len(gs1),
                        term2,
                        len(gs2),
                        term3,
                        len(gs3),
                        term4,
                        len(gs4),
                        k1234,
                    )

                    if len(heap) < top_k:
                        heapq.heappush(heap, record)
                    elif record > heap[0]:
                        heapq.heapreplace(heap, record)

    # final sorting: p-value ↑, jaccard ↓, intersection ↓
    results = sorted(
        heap,
        key=lambda x: (-x[0], -x[1], -x[2])
    )

    return [
        [
            term1,
            len1,
            term2,
            len2,
            term3,
            len3,
            term4,
            len4,
            f"{np.exp(-pval):.6e}",
            jaccard,
            str(k),
            " ".join(sorted(genes)),
        ] for pval, jaccard, k, term1, len1, term2, len2, term3, len3, term4, len4, genes in results
    ]

In [31]:
cols = ["Term1", "Term1 Length","Term2", "Term2 Length","Term3", "Term3 Length","p-value","Jaccard","Overlap","Genes"]

## Crosing 1: IDG Drug Targets & GlyGen Glycans & HuBMAP ASCT+B

In [34]:
idg = load_gmt('cfgmts/idgtargets.gmt')
glygen = load_gmt('cfgmts/glygen_up.gmt')
asctb = load_gmt('cfgmts/hubmapasctbaugmented.gmt')

In [35]:
res = cross_3_gmts(idg, glygen, asctb)
pd.DataFrame(res)

100%|██████████| 888/888 [01:47<00:00,  8.28it/s]


,0,1,2,3,4,5,6,7,8,9
0,Ximelagatran,8,G08146BT,60,Periportal Region hepatocyte - Liver,102,9.676376e-08,0.018868,3,F11 F7 PROC
1,Nilotinib,72,G38663NM,75,Marginal Reticular Cell - Lymph Node,106,1.518447e-07,0.012245,3,NTRK2 PDGFRB TEK
2,Ximelagatran,8,G42358LZ,58,Periportal Region hepatocyte - Liver,102,3.857485e-07,0.020548,3,F11 F7 PROC
3,Ximelagatran,8,G00912UN,140,Periportal Region hepatocyte - Liver,102,1.004245e-06,0.013636,3,F11 PLG PROC
4,Sorafenib,122,G38663NM,75,Marginal Reticular Cell - Lymph Node,106,1.079603e-06,0.010239,3,NTRK2 PDGFRB TEK
...,...,...,...,...,...,...,...,...,...,...
4995,Pemetrexed,9,G71051TA,5,Serous Cell Of Epithelium Of trachea - Respira...,101,4.590909e-03,0.008850,1,FOLR1
4996,Norethisterone,7,G81413UE,7,Endocrine Lineage cell - Small Intestine,101,4.590909e-03,0.008850,1,SHBG
4997,Norethindrone Acetate,7,G81413UE,7,Endocrine Lineage cell - Small Intestine,101,4.590909e-03,0.008850,1,SHBG
4998,Telaprevir,10,G15956KF,5,Kidney Granular cell - Kidney,101,4.590909e-03,0.008850,1,PLG


In [36]:
resframe = pd.DataFrame(res)
resframe.columns = cols
resframe.to_csv('idg_glygen_asctbaugmented_crossing.tsv', sep='\t')
resframe

,Term1,Term1 Length,Term2,Term2 Length,Term3,Term3 Length,p-value,Jaccard,Overlap,Genes
0,Ximelagatran,8,G08146BT,60,Periportal Region hepatocyte - Liver,102,9.676376e-08,0.018868,3,F11 F7 PROC
1,Nilotinib,72,G38663NM,75,Marginal Reticular Cell - Lymph Node,106,1.518447e-07,0.012245,3,NTRK2 PDGFRB TEK
2,Ximelagatran,8,G42358LZ,58,Periportal Region hepatocyte - Liver,102,3.857485e-07,0.020548,3,F11 F7 PROC
3,Ximelagatran,8,G00912UN,140,Periportal Region hepatocyte - Liver,102,1.004245e-06,0.013636,3,F11 PLG PROC
4,Sorafenib,122,G38663NM,75,Marginal Reticular Cell - Lymph Node,106,1.079603e-06,0.010239,3,NTRK2 PDGFRB TEK
...,...,...,...,...,...,...,...,...,...,...
4995,Pemetrexed,9,G71051TA,5,Serous Cell Of Epithelium Of trachea - Respira...,101,4.590909e-03,0.008850,1,FOLR1
4996,Norethisterone,7,G81413UE,7,Endocrine Lineage cell - Small Intestine,101,4.590909e-03,0.008850,1,SHBG
4997,Norethindrone Acetate,7,G81413UE,7,Endocrine Lineage cell - Small Intestine,101,4.590909e-03,0.008850,1,SHBG
4998,Telaprevir,10,G15956KF,5,Kidney Granular cell - Kidney,101,4.590909e-03,0.008850,1,PLG


## Crossing 2: IDG Drugs & IMPC Phenotype & MoTrPAC Exercise

In [24]:
idg = load_gmt('cfgmts/idgtargets.gmt')

In [25]:
impc = load_gmt('cfgmts/komp_up.gmt')
impc_trimmed = {term:geneset for term,geneset in impc.items() if len(geneset)<= 1000}

In [26]:
motrpac = load_gmt('cfgmts/motrpac_up.gmt', '_up') | load_gmt('cfgmts/motrpac_dn.gmt', '_dn')
motrpac_trimmed = {term:geneset for term,geneset in motrpac.items() if len(geneset)<= 1000}

In [27]:
res2 = cross_3_gmts(idg, impc_trimmed, motrpac_trimmed)
pd.DataFrame(res2)

100%|██████████| 888/888 [00:19<00:00, 46.21it/s] 


,0,1,2,3,4,5,6,7,8,9
0,Nifedipine,37,increased circulating aspartate transaminase l...,215,t60-adrenal_female_8w_dn,107,9.506630e-05,0.005666,2,ABCB11 ADORA2A
1,Metformin,48,embryonic lethality prior to organogenesis,534,t69-brown-adipose_male_8w_dn,338,1.260833e-04,0.006952,6,NDUFA6 NDUFA9 NDUFS2 NDUFS7 NDUFS8 NDUFV1
2,Primaquine,11,male infertility,289,t60-adrenal_male_8w_dn,14,1.719748e-04,0.006452,2,HSP90AA1 MAOB
3,Aclarubicin,5,abnormal seminal vesicle morphology,155,t69-brown-adipose_male_8w_up,341,2.395563e-04,0.004024,2,MMP2 TOP2A
4,Daunorubicin,6,abnormal seminal vesicle morphology,155,t69-brown-adipose_male_8w_up,341,2.395563e-04,0.004016,2,MMP2 TOP2A
...,...,...,...,...,...,...,...,...,...,...
4995,Vardenafil,14,decreased blood urea nitrogen level,55,t52-hippocampus_consensus_up,228,1.036364e-02,0.003413,1,PDE1A
4996,Papaverine,16,decreased blood urea nitrogen level,55,t52-hippocampus_consensus_up,228,1.036364e-02,0.003390,1,PDE1A
4997,Theophylline,12,decreased circulating triglyceride level,59,t52-hippocampus_consensus_up,228,1.036364e-02,0.003378,1,PDE1A
4998,Disulfiram,36,small superior vagus ganglion,35,t52-hippocampus_consensus_up,228,1.036364e-02,0.003367,1,ALOX15


In [28]:
resframe = pd.DataFrame(res2)
resframe.columns = cols
resframe.to_csv('idg_impc_motrpac.tsv', sep='\t')
resframe

,Term1,Term1 Length,Term2,Term2 Length,Term3,Term3 Length,p-value,Jaccard,Overlap,Genes
0,Nifedipine,37,increased circulating aspartate transaminase l...,215,t60-adrenal_female_8w_dn,107,9.506630e-05,0.005666,2,ABCB11 ADORA2A
1,Metformin,48,embryonic lethality prior to organogenesis,534,t69-brown-adipose_male_8w_dn,338,1.260833e-04,0.006952,6,NDUFA6 NDUFA9 NDUFS2 NDUFS7 NDUFS8 NDUFV1
2,Primaquine,11,male infertility,289,t60-adrenal_male_8w_dn,14,1.719748e-04,0.006452,2,HSP90AA1 MAOB
3,Aclarubicin,5,abnormal seminal vesicle morphology,155,t69-brown-adipose_male_8w_up,341,2.395563e-04,0.004024,2,MMP2 TOP2A
4,Daunorubicin,6,abnormal seminal vesicle morphology,155,t69-brown-adipose_male_8w_up,341,2.395563e-04,0.004016,2,MMP2 TOP2A
...,...,...,...,...,...,...,...,...,...,...
4995,Vardenafil,14,decreased blood urea nitrogen level,55,t52-hippocampus_consensus_up,228,1.036364e-02,0.003413,1,PDE1A
4996,Papaverine,16,decreased blood urea nitrogen level,55,t52-hippocampus_consensus_up,228,1.036364e-02,0.003390,1,PDE1A
4997,Theophylline,12,decreased circulating triglyceride level,59,t52-hippocampus_consensus_up,228,1.036364e-02,0.003378,1,PDE1A
4998,Disulfiram,36,small superior vagus ganglion,35,t52-hippocampus_consensus_up,228,1.036364e-02,0.003367,1,ALOX15


## Crosing 3: GTEx Aging & MoTrPAC Exercise & RummaGEO Gene Perturbations

In [30]:
def load_rummageo_gmt(file):
    gmt = {}
    with open(file, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            if len(line.strip().split('\t'))==1: continue
            term, _, *geneset = line.strip().split('\t')
            if geneset:
                gmt[term] = set(geneset)
    return gmt

In [145]:
rummageogene = load_rummageo_gmt('cfgmts/rummageogene.gmt')

In [146]:
res3 = cross_3_gmts(rummageogene, gtexaging, motrpac)
pd.DataFrame(res3)

100%|██████████| 4312/4312 [59:06<00:00,  1.22it/s]


,0,1,2,3,4,5,6,7,8,9
0,GSE168536_PERK_knockout_heart_2_v_3_mouse_up,1414,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,3.036557e-40,0.024448,41,AIFM1 ATP5PB C1QBP CHCHD3 CISD1 DLAT DLD DLST ...
1,GSE184986_Fap_inhibition_0_v_2_mouse_up,1028,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,2.647527e-38,0.028443,38,AIFM1 ATP5PB C1QBP CHCHD3 CISD1 DLAT DLD DLST ...
2,GSE99522_Tmem135_overexpression_1_v_0_mouse_down,423,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,3.051654e-37,0.038369,32,AIFM1 ATP5PB C1QBP CHCHD3 DLAT DLD DLST ETFA F...
3,GSE159187_Creld1_knockout_4_v_0_mouse_down,844,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,1.886884e-35,0.027094,33,ATP5PB CHCHD3 CISD1 DLAT DLD DLST ETFA FH GFM1...
4,GSE207717_GSE207718_FAK_inhibition_0_v_1_mouse...,959,GTEx Ovary 20-29 vs 50-59_up,249,t30-blood-rna_consensus_up,1369,2.449742e-33,0.021332,49,CETN2 CHCHD3 COPS9 FAU HINT1 HSP90AB1 NDUFAF8 ...
...,...,...,...,...,...,...,...,...,...,...
4995,GSE233566_Bmp8a adipose_overexpression_3T3-L1_...,917,GTEx Lung 20-29 vs 50-59_dn,249,t61-colon_consensus_up,2279,1.416465e-09,0.005722,18,AIFM1 APEX1 CHAF1A EIF6 FH GRHPR HADHA IMMT IT...
4996,GSE252944_AKT_inhibition_10_v_15_human_down,954,GTEx Lung 20-29 vs 50-59_dn,249,t61-colon_consensus_up,2279,1.416465e-09,0.005680,18,ATP5F1B AURKB CYC1 DDOST FH IMMT IPO4 MDH2 MRP...
4997,GSE193993_SSTR2_knockout overexpression_4_v_7_...,1822,GTEx Ovary 20-29 vs 50-59_up,249,t59-kidney_consensus_up,282,1.418839e-09,0.006781,15,FAU RPL17 RPL21 RPL26 RPL32 RPL37A RPL4 RPL41 ...
4998,GSE190178_CDK8 MEK_inhibition_3_v_2_human_down,1714,GTEx AdiposeTissue 20-29 vs 60-69_dn,249,t60-adrenal_consensus_up,4015,1.418884e-09,0.005837,31,ACOT2 AK2 ATP5F1B ATP5MC3 COPS3 COX5A CRK CS D...


In [163]:
resframe = pd.DataFrame(res3)
resframe.columns = cols
resframe.to_csv('rummageogene_gtexaging_motrpac_crossing.tsv', sep='\t')
resframe

,Term1,Term1 Length,Term2,Term2 Length,Term3,Term3 Length,p-value,Jaccard,Overlap,Genes
0,GSE168536_PERK_knockout_heart_2_v_3_mouse_up,1414,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,3.036557e-40,0.024448,41,AIFM1 ATP5PB C1QBP CHCHD3 CISD1 DLAT DLD DLST ...
1,GSE184986_Fap_inhibition_0_v_2_mouse_up,1028,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,2.647527e-38,0.028443,38,AIFM1 ATP5PB C1QBP CHCHD3 CISD1 DLAT DLD DLST ...
2,GSE99522_Tmem135_overexpression_1_v_0_mouse_down,423,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,3.051654e-37,0.038369,32,AIFM1 ATP5PB C1QBP CHCHD3 DLAT DLD DLST ETFA F...
3,GSE159187_Creld1_knockout_4_v_0_mouse_down,844,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,1.886884e-35,0.027094,33,ATP5PB CHCHD3 CISD1 DLAT DLD DLST ETFA FH GFM1...
4,GSE207717_GSE207718_FAK_inhibition_0_v_1_mouse...,959,GTEx Ovary 20-29 vs 50-59_up,249,t30-blood-rna_consensus_up,1369,2.449742e-33,0.021332,49,CETN2 CHCHD3 COPS9 FAU HINT1 HSP90AB1 NDUFAF8 ...
...,...,...,...,...,...,...,...,...,...,...
4995,GSE233566_Bmp8a adipose_overexpression_3T3-L1_...,917,GTEx Lung 20-29 vs 50-59_dn,249,t61-colon_consensus_up,2279,1.416465e-09,0.005722,18,AIFM1 APEX1 CHAF1A EIF6 FH GRHPR HADHA IMMT IT...
4996,GSE252944_AKT_inhibition_10_v_15_human_down,954,GTEx Lung 20-29 vs 50-59_dn,249,t61-colon_consensus_up,2279,1.416465e-09,0.005680,18,ATP5F1B AURKB CYC1 DDOST FH IMMT IPO4 MDH2 MRP...
4997,GSE193993_SSTR2_knockout overexpression_4_v_7_...,1822,GTEx Ovary 20-29 vs 50-59_up,249,t59-kidney_consensus_up,282,1.418839e-09,0.006781,15,FAU RPL17 RPL21 RPL26 RPL32 RPL37A RPL4 RPL41 ...
4998,GSE190178_CDK8 MEK_inhibition_3_v_2_human_down,1714,GTEx AdiposeTissue 20-29 vs 60-69_dn,249,t60-adrenal_consensus_up,4015,1.418884e-09,0.005837,31,ACOT2 AK2 ATP5F1B ATP5MC3 COPS3 COX5A CRK CS D...


In [4]:
rummageo_mouse_meta = json.load(open('meta/mouse-gse-processed-meta.json', 'r'))

In [8]:
print(
    rummageo_mouse_meta['GSE168536']['titles']['2'],
    rummageo_mouse_meta['GSE168536']['titles']['3'],
    sep='\n'
)

sample flx heart wild type developmental sgtage nulliparous mouse
sample cko heart perk mutant developmental sgtage postpartum mouse


## Crosing 4: GTEx Aging & LINCS Drugs & MoTrPAC Exercise & Metabolomics Metabolites

In [15]:
l1000drug = load_gmt('cfgmts/l1000_cp_pruned_250_up.gmt', '_up') | load_gmt('cfgmts/l1000_cp_pruned_250_down.gmt', '_dn')
gtexaging = load_gmt('cfgmts/gtexagingsignatures_up.gmt', '_up') | load_gmt('cfgmts/gtexagingsignatures_dn.gmt', '_dn')
motrpac = load_gmt('cfgmts/motrpac_up.gmt', '_up') | load_gmt('cfgmts/motrpac_dn.gmt', '_dn')
mwmetabolites = load_gmt('cfgmts/mwmetabolites_up.gmt')

In [16]:
res4 = cross_4_gmts(l1000drug, gtexaging, motrpac, mwmetabolites)

100%|██████████| 156926/156926 [24:37:07<00:00,  1.77it/s]   


In [17]:
cols_4 = ["Term1", "Term1 Length","Term2", "Term2 Length","Term3", "Term3 Length","Term4", "Term4 Length","p-value","Jaccard","Overlap","Genes"]

In [18]:
res_4_df = pd.DataFrame(res4, columns=cols_4)
res_4_df.to_csv('l1000drug_gtexaging_motrpac_mwmetabolites.tsv', sep='\t')
res_4_df

,Term1,Term1 Length,Term2,Term2 Length,Term3,Term3 Length,Term4,Term4 Length,p-value,Jaccard,Overlap,Genes
0,CPC011_VCAP_24H_F16_loteprednol_10uM_dn,250,GTEx AdiposeTissue 20-29 vs 50-59_dn,249,t69-brown-adipose_male_8w_dn,338,NAD+,105,4.924910e-09,0.005841,5,DLAT DLST IDH3A NDUFV2 PDHA1
1,EMU001_OCILY3_24H_F01_sirolimus_10uM_dn,250,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,Coenzyme A,98,9.916252e-09,0.005952,5,DLD DLST HADHB SUCLA2 SUCLG1
2,REP.A019_YAPC_24H_A08_taselisib_3.33uM_dn,250,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,Coenzyme A,98,9.916252e-09,0.005945,5,DLAT DLD DLST SUCLA2 SUCLG1
3,PBIOA017_PC3_24H_K10_troglitazone_10uM_up,250,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,Coenzyme A,98,9.916252e-09,0.005896,5,DLD DLST PDHA1 SUCLA2 SUCLG1
4,CPC012_PC3_24H_G18_BRD-K73155123_10uM_dn,250,GTEx AdiposeTissue 20-29 vs 50-59_dn,249,t60-adrenal_male_1w_up,318,Acetyl-CoA,36,4.284614e-08,0.005076,4,CS DLAT FASN PDHA1
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,POL001_HT29_24H_N01_daunorubicin_10uM_up,250,GTEx Heart 20-29 vs 60-69_dn,249,t58-heart_female_8w_up,21,NAD+,105,1.286215e-04,0.003322,2,DLD NDUFS1
4996,REP.A024_A375_24H_F18_procainamide_0.04uM_up,250,GTEx Thyroid 20-29 vs 70-79_dn,249,t58-heart_female_8w_up,21,NAD+,105,1.286215e-04,0.003322,2,DLD NDUFS1
4997,REP.A024_A375_24H_F18_procainamide_0.04uM_up,250,GTEx Heart 20-29 vs 70-79_dn,249,t58-heart_female_8w_up,21,NAD+,105,1.286215e-04,0.003322,2,DLD NDUFS1
4998,PCLB003_HEPG2_24H_P13_doxorubicin_10uM_up,250,GTEx Heart 20-29 vs 60-69_dn,249,t58-heart_female_8w_up,21,NAD+,105,1.286215e-04,0.003322,2,DLD NDUFS1


## Crossing 5: RummaGEO Drug x GTEx x MoTrPAC x Metabolomics

In [31]:
rummageo_drug = load_rummageo_gmt('cfgmts/rummageodrug.gmt')
gtexaging = load_gmt('cfgmts/gtexagingsignatures_up.gmt', '_up') | load_gmt('cfgmts/gtexagingsignatures_dn.gmt', '_dn')
motrpac = load_gmt('cfgmts/motrpac_up.gmt', '_up') | load_gmt('cfgmts/motrpac_dn.gmt', '_dn')
mwmetabolites = load_gmt('cfgmts/mwmetabolites_up.gmt')

In [32]:
res5 = cross_4_gmts(rummageo_drug, gtexaging, motrpac, mwmetabolites)

100%|██████████| 5344/5344 [1:26:39<00:00,  1.03it/s]


In [33]:
res5_df = pd.DataFrame(res5, columns=cols_4)
res5_df.to_csv('rummageodrug_gtexaging_motrpac_mwmetabolites.tsv', sep='\t')
res5_df

,Term1,Term1 Length,Term2,Term2 Length,Term3,Term3 Length,Term4,Term4 Length,p-value,Jaccard,Overlap,Genes
0,GSE255432 4 V 0 IMEGLIMIN MOUSE UP,1422,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_consensus_up,1470,NAD+,105,1.144562e-13,0.004095,11,DLAT DLD DLST IDH3B LDHB MDH1 MDH2 NDUFS1 NDUF...
1,GSE179531 1 V 0 METFORMIN MOUSE UP,1586,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_consensus_up,1470,NAD+,105,2.646534e-13,0.003868,11,DLAT DLD DLST IDH3B LDHB MDH1 MDH2 NDUFS1 NDUF...
2,"GSE226114,GSE226116 2 V 0 DOXORUBICIN MOUSE DN",920,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,NAD+,105,3.766607e-13,0.006886,9,DLAT DLD DLST IDH3B MDH2 NDUFS1 NDUFS2 PDHA1 PDHB
3,GSE255432 7 V 2 IMEGLIMIN MOUSE DN,644,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_male_8w_dn,338,NAD+,105,4.462835e-13,0.008654,9,DLAT DLD DLST IDH3B MDH2 NDUFS1 NDUFS2 PDHA1 PDHB
4,"GSE226114,GSE226116 1 V 0 DOXORUBICIN MOUSE DN",1666,GTEx Heart 20-29 vs 60-69_dn,249,t69-brown-adipose_consensus_up,1470,NAD+,105,4.555056e-13,0.003715,11,DLAT DLD DLST IDH3B LDHB MDH1 MDH2 NDUFS1 NDUF...
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,GSE158834 1 V 9 PALMITIC ACID HUMAN DN,1043,GTEx AdiposeTissue 20-29 vs 30-39_dn,249,t69-brown-adipose_consensus_up,1470,Acetyl-CoA,36,2.977532e-04,0.001193,3,CS DLAT PDHA1
4996,GSE158834 1 V 9 PALMITIC ACID HUMAN DN,1043,GTEx AdiposeTissue 20-29 vs 60-69_dn,249,t69-brown-adipose_consensus_up,1470,Pyruvic acid,24,2.977532e-04,0.001193,3,DLAT LDHB PDHA1
4997,GSE126463 0 V 3 POMALIDOMIDE HUMAN DN,924,GTEx Colon 20-29 vs 60-69_dn,249,t69-brown-adipose_consensus_up,1470,NAD+,105,2.977532e-04,0.001191,3,DLAT IDH3A NDUFS1
4998,GSE210655 1 V 2 CISPLATIN MOUSE UP,1059,GTEx Esophagus 20-29 vs 70-79_dn,249,t69-brown-adipose_consensus_up,1470,Succinyl-CoA,10,2.977532e-04,0.001190,3,DLD MMUT SUCLG2


## Crosing 6: LINCS CRISPR & GTEx Aging & Glygen Glycosylated Proteins

In [20]:
l1000crispr = load_gmt('cfgmts/l1000_xpr_pruned_250.gmt')
gtexaging = load_gmt('cfgmts/gtexagingsignatures_up.gmt', '_up') | load_gmt('cfgmts/gtexagingsignatures_dn.gmt', '_dn')
glygen = load_gmt('cfgmts/glygen_up.gmt')

In [22]:
res6 = cross_3_gmts(l1000crispr, gtexaging, glygen)

100%|██████████| 65672/65672 [16:40:09<00:00,  1.09it/s]   


In [ ]:
cols_6 = ["Term1", "Term1 Length","Term2", "Term2 Length","Term3", "Term3 Length","Term4", "Term4 Length","p-value","Jaccard","Overlap","Genes"]

In [23]:
res_6_df = pd.DataFrame(res6, columns=cols)
res_6_df.to_csv('l1000xpr_gtexaging_glygen.tsv', sep='\t')
res_6_df

,Term1,Term1 Length,Term2,Term2 Length,Term3,Term3 Length,p-value,Jaccard,Overlap,Genes
0,XPR031_ES2.311_96H_P24_PPP1R16B down,250,GTEx Brain 20-29 vs 60-69_up,249,G82830MN,164,5.924912e-14,0.014446,9,AZGP1 BGN CFB CFH CFI CLU CP PTGDS SERPINA3
1,XPR015_YAPC.311_96H_F13_DCLK2 down,250,GTEx Muscle 20-29 vs 60-69_up,249,G43417UB,605,7.001954e-14,0.011673,12,BGN COL6A2 CP ELN FBLN2 FBLN5 ISLR LUM MFAP2 M...
2,XPR035_MCF7.311_96H_B08_PRKD1 up,250,GTEx Brain 20-29 vs 60-69_up,249,G00912UN,140,6.815310e-13,0.013115,8,C1S CFB CFH CFI CLU CP SERPINA3 VWF
3,XPR027_AGS.311_96H_I10_VSIG1 down,250,GTEx Muscle 20-29 vs 60-69_up,249,G43417UB,605,1.036439e-12,0.009615,10,BGN COL6A2 ELN FBLN2 ISLR LUM MFAP2 PCOLCE SEM...
4,XPR016_A375.311_96H_J10_FAIM2 down,250,GTEx Muscle 20-29 vs 70-79_up,249,G05724UK,262,1.153982e-12,0.012465,9,C2 C3 CD74 CLU HLA-DPA1 HLA-DRA MPZL2 PLAT PLXNB2
...,...,...,...,...,...,...,...,...,...,...
4995,XPR042_IPC298.311_96H_D10_BRDN0003228153 up,250,GTEx Brain 20-29 vs 60-69_up,249,G85144OK,33,4.293143e-08,0.009823,5,AZGP1 CFH CLU CP SERPINA3
4996,XPR021_BICR6.311_96H_M03_MXRA5 up,250,GTEx Brain 20-29 vs 60-69_up,249,G85144OK,33,4.293143e-08,0.009804,5,AZGP1 CFH CFI CP SERPINA3
4997,XPR025_A375.311_96H_A14_STARD3 up,250,GTEx Brain 20-29 vs 60-69_up,249,G85144OK,33,4.293143e-08,0.009804,5,AZGP1 CFH CFI CP SERPINA3
4998,XPR040_LCLC103H.311_96H_B16_RFX5 down,250,GTEx Brain 20-29 vs 70-79_up,249,G85144OK,33,4.293143e-08,0.009804,5,AZGP1 CFH CFI CP SERPINA3
